# Наївний Баєсів класифікатор (Naïve Bayes) та NLP

In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

## Обробка тексту та виділення ознак

In [2]:
def load_data(pos_path, neg_path):
    with open(pos_path, 'r', encoding='utf-8', errors='ignore') as f:
        pos_text = f.read().splitlines()
    with open(neg_path, 'r', encoding='utf-8', errors='ignore') as f:
        neg_text = f.read().splitlines()
    
    X = pos_text + neg_text
    y = [1] * len(pos_text) + [0] * len(neg_text)
    return train_test_split(X, y, test_size=0.2, random_state=42)

X_train, X_test, y_train, y_test = load_data('rt-polarity.pos', 'rt-polarity.neg')
print(f"Дані завантажено. Тренувальна вибірка: {len(X_train)} відгуків")

Дані завантажено. Тренувальна вибірка: 8529 відгуків


## Класифікація тексту та оцінка якості моделей

In [3]:
text_logistic_clf = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))),
    ('clf-logistic', LogisticRegression())
])

text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))),
    ('clf', MultinomialNB())
])

text_svm_clf = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))),
    ('clf-svm', SVC(kernel='linear', C=1.0, probability=True))
])

text_logistic_clf.fit(X_train, y_train)
text_clf.fit(X_train, y_train)
text_svm_clf.fit(X_train, y_train)

predictions_logistic = text_logistic_clf.predict(X_test)
predictions = text_clf.predict(X_test)
predictions_svm = text_svm_clf.predict(X_test)

print("\n--- Звіт про класифікацію TF-IDF + Logistic Regression ---")
print(classification_report(y_test, predictions_logistic, target_names=['Negative', 'Positive']))

print("\n--- Звіт про класифікацію TF-IDF + Naive Bayes ---")
print(classification_report(y_test, predictions, target_names=['Negative', 'Positive']))

print("\n--- Звіт про класифікацію TF-IDF + SVM ---")
print(classification_report(y_test, predictions_svm, target_names=['Negative', 'Positive']))


--- Звіт про класифікацію TF-IDF + Logistic Regression ---
              precision    recall  f1-score   support

    Negative       0.73      0.75      0.74      1071
    Positive       0.74      0.72      0.73      1062

    accuracy                           0.74      2133
   macro avg       0.74      0.74      0.74      2133
weighted avg       0.74      0.74      0.74      2133


--- Звіт про класифікацію TF-IDF + Naive Bayes ---
              precision    recall  f1-score   support

    Negative       0.75      0.75      0.75      1071
    Positive       0.75      0.75      0.75      1062

    accuracy                           0.75      2133
   macro avg       0.75      0.75      0.75      2133
weighted avg       0.75      0.75      0.75      2133


--- Звіт про класифікацію TF-IDF + SVM ---
              precision    recall  f1-score   support

    Negative       0.73      0.75      0.74      1071
    Positive       0.74      0.72      0.73      1062

    accuracy              

## Ітого
Logistic Regression - **74%**, Naive Bayes - **75%**, SVM - **73%**